In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
import jax.numpy as jnp

from hcnn.solver.solver_osqp import (
    OSQPParameters,
    OSQPProblem,
    osqp_solve,
)

A = jnp.array([[1.0, 0.0], [1.0, 0.0]])


def matrix_vector_product(x):
    return A @ x


def vector_matrix_product(y):
    return A.T @ y


osqp_problem = OSQPProblem(
    x0=jnp.array([0.0, 0.0]),
    cost_quad_diag=jnp.array([1.0, 1.0]),
    cost_linear=jnp.array([0.0, 0.0]),
    lower_bound=jnp.array([0.5, 1.5]),
    upper_bound=jnp.array([1.0, 2.0]),
    matrix_vector_product=matrix_vector_product,
    vector_matrix_product=vector_matrix_product,
)

osqp_variables = osqp_solve(
    problem=osqp_problem,
    parameters=OSQPParameters(),
)

print(osqp_variables)

OSQPVariables(x_tilde=Array([1.2500072, 0.       ], dtype=float32), z_tilde=Array([1.2500072, 1.2500072], dtype=float32), x=Array([1.2499981, 0.       ], dtype=float32), z=Array([1. , 1.5], dtype=float32), y=Array([ 956824.06, -956825.  ], dtype=float32), max_primal_residual=Array(0.2500019, dtype=float32), max_dual_residual=Array(0.3124981, dtype=float32), rho_base_mult=Array(131072., dtype=float32), iteration=Array(580, dtype=int32, weak_type=True), is_infeasible=Array(True, dtype=bool))


In [3]:
import jax
import jax.numpy as jnp
from flax import struct

from hcnn.solver.solver_sqp import (
    SQPParameters,
    SQPProblemBase,
    sqp_solve,
)


@struct.dataclass
class SimpleProblem(SQPProblemBase[jax.Array]):
    def get_x0(self, phi):
        return jnp.zeros_like(self.cost_quad_diag)

    def constraint_fn_phi(self, x: jax.Array, phi: jax.Array) -> jax.Array:
        return phi * x

    @staticmethod
    def factory():
        return SimpleProblem(
            cost_quad_diag=jnp.array([1.0, 1.0]),
            cost_linear=jnp.array([0.0, 0.0]),
            phi=jnp.array([1.0, 1.0]),
            lower_bound=jnp.array([0.5, 0.5]),
            upper_bound=jnp.array([1.0, 1.0]),
        )


sqp_variables = sqp_solve(
    SimpleProblem.factory(),
    SQPParameters(),
)

print(sqp_variables)

(Array([0.49996188, 0.49996188], dtype=float32), SQPInfo(primal_residual=Array(3.811717e-05, dtype=float32), dual_residual=Array(8.6426735e-07, dtype=float32), iterations_lin=Array(40., dtype=float32, weak_type=True), iterations_nonlin=Array(1, dtype=int32, weak_type=True), rho_base_mult=Array(4., dtype=float32), converged_flag=Array(True, dtype=bool), avg_step_size=Array(1., dtype=float32, weak_type=True)))


In [4]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.test_util import check_vjp

phi_test = jnp.array([1.0, 1.0])


# Define a simple test function that uses sqp_solve
def test_function(phi):
    problem = SimpleProblem.factory()
    result, info = sqp_solve(
        problem.replace(phi=phi),
        SQPParameters(),
    )
    return result


check_vjp(
    f=test_function,
    f_vjp=partial(jax.vjp, test_function),
    args=(phi_test,),
)

In [5]:
@struct.dataclass
class QuadProblem(SQPProblemBase[jax.Array]):
    def get_x0(self, phi):
        return jnp.array([2.0, 2.0])

    def constraint_fn_phi(self, x: jax.Array, phi: jax.Array) -> jax.Array:
        return jnp.array(
            [
                x[0],
                x[1],
                phi[0] * x[0] ** 2 - phi[1] * x[1],
                jnp.sin(x[0]) + 1 / 2 - x[1],
            ]
        )

    @staticmethod
    def factory():
        return QuadProblem(
            cost_quad_diag=jnp.array([1.0, 1.0]),
            cost_linear=jnp.array([0.0, 0.0]),
            phi=jnp.array([1.0, 1.0]),
            lower_bound=jnp.array([1.0, 1.0, 0.0, 0]),
            upper_bound=jnp.array([2.0, 2.0, 0.0, 0]),
        )


sqp_variables = sqp_solve(
    QuadProblem.factory(),
    SQPParameters(),
)

print(sqp_variables)

(Array([1.196083 , 1.4306114], dtype=float32), SQPInfo(primal_residual=Array(3.0994415e-06, dtype=float32), dual_residual=Array(4.708767e-05, dtype=float32), iterations_lin=Array(52., dtype=float32, weak_type=True), iterations_nonlin=Array(5, dtype=int32, weak_type=True), rho_base_mult=Array(0.353125, dtype=float32), converged_flag=Array(True, dtype=bool), avg_step_size=Array(1., dtype=float32, weak_type=True)))


In [6]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.test_util import check_vjp

phi_test = jnp.array([1.0, 1.0])


def test_function(phi):
    problem = QuadProblem.factory()
    x, info = sqp_solve(
        problem.replace(phi=phi),
        SQPParameters(),
    )
    return x  # primal solution only


check_vjp(
    f=test_function,
    f_vjp=partial(jax.vjp, test_function),
    args=(phi_test,),
)